# 07 · Checkpointer vs. Store

Agents are stateless — every `invoke` starts blank. LangGraph gives two memory primitives:

```
CHECKPOINTER  →  the transcript of ONE conversation (thread-scoped, automatic)
STORE         →  facts that outlive conversations   (cross-thread, explicit)
```

Checkpointer = "what were we just talking about?" · Store = "what do I know about this user, ever?"

---

## Setup

In [ ]:
%pip install -qU langchain langchain-openai python-dotenv

In [ ]:
import os
from uuid import uuid4

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.config import get_store
from dotenv import load_dotenv

load_dotenv("../../.env")

llm = init_chat_model(os.environ["CHAT_MODEL"], temperature=0)

---
## 1. The problem: agents are goldfish

No memory — the second call has no idea what happened in the first.

In [ ]:
forgetful = create_agent(llm, tools=[])

forgetful.invoke({"messages": [("human", "Hi! My name is Harish and I live in Bangalore.")]})
out = forgetful.invoke({"messages": [("human", "What's my name?")]})
print(out["messages"][-1].content)

---
## 2. Checkpointer: memory within a thread

Add `checkpointer=` and pass a `thread_id` in the config. Send **only the new message** — loading and saving the transcript is automatic.

In [ ]:
agent = create_agent(llm, tools=[], checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "chat-1"}}

agent.invoke({"messages": [("human", "Hi! My name is Harish and I live in Bangalore.")]}, config)
out = agent.invoke({"messages": [("human", "What's my name and where do I live?")]}, config)
print(out["messages"][-1].content)

A **different thread_id** is a clean slate — same agent, separate transcript.

In [ ]:
other = {"configurable": {"thread_id": "chat-2"}}
out = agent.invoke({"messages": [("human", "What's my name?")]}, other)
print(out["messages"][-1].content)

### Peek at what the checkpointer saved

In [ ]:
state = agent.get_state(config)
print(f"Messages stored in thread 'chat-1': {len(state.values['messages'])}")
for m in state.values["messages"]:
    print(f"  {m.type:>6}: {m.content[:70]}")

---
## 3. Store: memory across threads

A store is a namespaced key→value memory, independent of any thread. Namespaces are tuples — think folder paths like `("memories", user_id)`.

In [ ]:
demo_store = InMemoryStore()
ns = ("memories", "harish")

demo_store.put(ns, "pref-units", {"fact": "prefers metric units"})
demo_store.put(ns, "pref-diet", {"fact": "is vegetarian"})

print(demo_store.get(ns, "pref-units").value)
print([item.value["fact"] for item in demo_store.search(ns)])

---
## 4. An agent that remembers you across conversations

Pass `store=` to the agent; tools access it at runtime with `get_store()`. Now the *model* decides when to remember and when to recall.

In [ ]:
@tool
def save_memory(fact: str) -> str:
    """Save a fact about the user so it can be recalled in future conversations."""
    get_store().put(("memories", "user"), str(uuid4()), {"fact": fact})
    return f"Saved: {fact}"

@tool
def recall_memories() -> str:
    """Recall all facts saved about the user from past conversations."""
    items = get_store().search(("memories", "user"))
    return "\n".join(i.value["fact"] for i in items) or "No memories saved yet."

memory_agent = create_agent(
    llm,
    [save_memory, recall_memories],
    system_prompt="When the user shares a lasting preference or fact about themselves, "
                  "save it. When asked about the user, recall memories first.",
    checkpointer=InMemorySaver(),
    store=InMemoryStore(),
)

In [ ]:
# Conversation A — the user shares a preference.
thread_a = {"configurable": {"thread_id": "monday"}}
out = memory_agent.invoke(
    {"messages": [("human", "Please remember: I prefer metric units and short answers.")]},
    thread_a,
)
print(out["messages"][-1].content)

In [ ]:
# Conversation B — a brand-new thread. The checkpointer knows nothing here,
# but the store does.
thread_b = {"configurable": {"thread_id": "friday"}}
out = memory_agent.invoke(
    {"messages": [("human", "How far is a marathon? Answer the way I like it.")]},
    thread_b,
)
print(out["messages"][-1].content)

> Thread B's transcript was empty — the answer came from the **store** (`recall_memories`), then respected the saved preference (km, short). That's cross-thread memory.

---
## What to remember

| Concept | What it does |
|---|---|
| `checkpointer=InMemorySaver()` | Auto-saves the transcript per `thread_id` — send only the new message |
| `{"configurable": {"thread_id": ...}}` | Selects which conversation to continue |
| `agent.get_state(config)` | Inspect the saved state |
| `store=InMemoryStore()` + `get_store()` | Cross-thread, namespaced memory that tools read/write |
| `SqliteSaver` / `PostgresSaver` | Drop-in persistent checkpointers for production |

**Checkpointer** = the chat transcript. **Store** = the user-profile database. Production agents run both.

**Next:** 08 — **Conversation memory patterns**: the transcript grows every turn — trimming, windowing, and summarization decide what the model actually *sees*.